- 这里尝试不同懒加载实现方案的导入

- 顺带一下`importlib.import_module`的使用，在下面这个例子里，就相当于执行了`import math`

In [1]:
import importlib

# 相当于 import math
math = importlib.import_module("math")
print(math.sqrt(16))  # 4.0

4.0


# PEP 562 使用 `__getattr__` 和 `__dir__`
- 执行 `import ModuleA` 时
> 1. 检查模块字典`sys.module`（如果存在，交出引用，否则创建新的空模块对象）
> 2. 执行`__init__.py`（导入内置的`importlib`库；创建 `_LAZY_IMPORTS` 字典；定义 `__getattr__` 函数；定义 `__dir__` 函数）
- `__getattr__` 的触发
> 1. 当试图访问一个“不存在”的属性时，它才会被触发（最开始为类设计）
> 2. 在这个例子里，当需要其他未被加载的模块被导入时，触发该函数
> 3. 该函数负责：导入子模块，将该子模块加入父模块的全局命名空间，后续不会重复加载
- `__dir__`
> 1. 自定义当调用 dir() 函数时返回的属性列表。它控制对象的属性和方法的可见性

In [10]:
import ModuleA
importlib.reload(ModuleA)
print(ModuleA.a)

<module 'ModuleA.A' from 'E:\\projects\\PyCharmProject\\rookie\\AboutPython\\LazyImport\\ModuleA\\A.py'>


In [11]:
globals()

{'__name__': '__main__',
 '__doc__': 'Automatically created module for IPython interactive environment',
 '__package__': None,
 '__loader__': None,
 '__spec__': None,
 '__builtin__': <module 'builtins' (built-in)>,
 '__builtins__': <module 'builtins' (built-in)>,
 '_ih': ['',
  'import importlib\n\n# 相当于 import math\nmath = importlib.import_module("math")\nprint(math.sqrt(16))  # 4.0',
  'import ModuleA\nprint(ModuleA.__dir__) ',
  'import ModuleA\nprint(ModuleA.__dir__) \nprint(ModuleA.__doc__) ',
  'import ModuleA\nprint(ModuleA.__dir__) \nprint(ModuleA.__doc__) \ndir(ModuleA)',
  'import ModuleA\nprint(ModuleA.__getattr__("a"))\nprint(ModuleA.__doc__)',
  'import ModuleA\nprint(ModuleA.__getattr__("A"))\nprint(ModuleA.__doc__)',
  'import ModuleA\nprint(ModuleA.__getattr__("a"))\nprint(ModuleA.__doc__)',
  'import ModuleA\nprint(ModuleA.a)',
  'import ModuleA\nprint(ModuleA.a)',
  'import ModuleA\nimportlib.reload(ModuleA)\nprint(ModuleA.a)',
  'globals()'],
 '_oh': {4: ['_LAZY_LOAD

- 这里我们检查一下加载前后多了哪些模块

In [13]:
import sys
del sys.modules["ModuleA"]
del sys.modules["ModuleA.A"]
# 1. 导入前：记录当前所有模块名
before = set(sys.modules)

import ModuleA

# 2. import ModuleA 之后
after_import = set(sys.modules)

print(ModuleA.a)   # 触发懒加载

# 3. 访问懒加载属性之后
after_lazy = set(sys.modules)

# 比较新增的模块名
print("import ModuleA 新增：")
for name in sorted(after_import - before):
    print("  ", name)

print("访问 ModuleA.a 新增：")
for name in sorted(after_lazy - after_import):
    print("  ", name)

print("总共新增：")
for name in sorted(after_lazy - before):
    print("  ", name)

<module 'ModuleA.A' from 'E:\\projects\\PyCharmProject\\rookie\\AboutPython\\LazyImport\\ModuleA\\A.py'>
import ModuleA 新增：
   ModuleA
访问 ModuleA.a 新增：
   ModuleA.A
总共新增：
   ModuleA
   ModuleA.A
